In [1]:
import pickle
from cs336_basics.bpe import BPE   # 你实现的 train_bpe

vocab, merges = BPE(
    input_path="data/TinyStoriesV2-GPT4-train.txt",
    vocab_size=10000,
    special_tokens=["<|endoftext|>"],
).train_bpe()

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
with open("merges.pkl", "wb") as f:
    pickle.dump(merges, f)

当前词表大小: 300, 当前进度：0.03
当前词表大小: 600, 当前进度：0.06
当前词表大小: 900, 当前进度：0.09
当前词表大小: 1200, 当前进度：0.12
当前词表大小: 1500, 当前进度：0.15
当前词表大小: 1800, 当前进度：0.18
当前词表大小: 2100, 当前进度：0.21
当前词表大小: 2400, 当前进度：0.24
当前词表大小: 2700, 当前进度：0.27
当前词表大小: 3000, 当前进度：0.3
当前词表大小: 3300, 当前进度：0.33
当前词表大小: 3600, 当前进度：0.36
当前词表大小: 3900, 当前进度：0.39
当前词表大小: 4200, 当前进度：0.42
当前词表大小: 4500, 当前进度：0.45
当前词表大小: 4800, 当前进度：0.48
当前词表大小: 5100, 当前进度：0.51
当前词表大小: 5400, 当前进度：0.54
当前词表大小: 5700, 当前进度：0.57
当前词表大小: 6000, 当前进度：0.6
当前词表大小: 6300, 当前进度：0.63
当前词表大小: 6600, 当前进度：0.66
当前词表大小: 6900, 当前进度：0.69
当前词表大小: 7200, 当前进度：0.72
当前词表大小: 7500, 当前进度：0.75
当前词表大小: 7800, 当前进度：0.78
当前词表大小: 8100, 当前进度：0.81
当前词表大小: 8400, 当前进度：0.84
当前词表大小: 8700, 当前进度：0.87
当前词表大小: 9000, 当前进度：0.9
当前词表大小: 9300, 当前进度：0.93
当前词表大小: 9600, 当前进度：0.96
当前词表大小: 9900, 当前进度：0.99


In [4]:
import numpy as np
from cs336_basics.tokenizer import Tokenizer

tokenizer = Tokenizer.from_files(
    vocab_filepath="vocab.pkl",
    merges_filepath="merges.pkl",
    special_tokens=["<|endoftext|>"],
)

def encode_file(in_path:str, out_path:str):
    with open(in_path, "r") as f:
        ids_iter = tokenizer.encode_iterable(f)
        arr = np.fromiter(ids_iter, dtype=np.uint16)
    np.save(out_path, arr)
    print(f"{out_path}: {len(arr)} tokens")

encode_file("data/TinyStoriesV2-GPT4-train.txt", "data/train.npy")
encode_file("data/TinyStoriesV2-GPT4-valid.txt", "data/val.npy")

data/train.npy: 563209460 tokens
data/val.npy: 5465871 tokens


uv run python cs336_basics/train.py \
    --train_data ./data/train.npy \
    --val_data ./data/val.npy \
    --vocab_path ./vocab.pkl \
    --merges_path ./merges.pkl \
    --checkpoint_path ./checkpoints \
    --device mps \
    --vocab_size 10000 \
    --context_length 256 \
    --d_model 512 \
    --num_layers 4 \
    --num_heads 16 \
    --d_ff 1344 \
    --batch_size 32 \
    --max_iters 5000 \
    --warmup_iters 200 \
    --lr_max 1e-3 \
    --lr_min 1e-4 \
    --weight_decay 0.01 \
    --grad_clip 1.0 \
    --log_interval 50 \
    --val_interval 200 \
    --save_interval 1000

tail -f checkpoints/train.log   # 如果 train.py 写了日志文件

In [1]:
import torch
from cs336_basics.model import TransformerLM
from cs336_basics.tokenizer import Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 分词器（路径换成你自己的）
tokenizer = Tokenizer.from_files(
    vocab_filepath="vocab.pkl",
    merges_filepath="merges.pkl",
    special_tokens=["<|endoftext|>"],
)
vocab_size = len(tokenizer.vocab)          # ← 注意：一定要用它，不要用训练时的 args.vocab_size

# 2. 模型结构必须和训练时完全一致
model = TransformerLM(
    vocab_size=vocab_size,
    context_length=256,      # 和训练时一样
    num_layers=4,
    d_model=512,
    num_heads=8,
    d_ff=1344,
    theta=10000.0,
).to(device)

# 3. 加载权重
ckpt = torch.load("checkpoints/latest.pt", map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

# 4. 编码 prompt → generate → decode
prompt_text = "Once upon a time"
ids = tokenizer.encode(prompt_text)
prompt = torch.tensor([ids], dtype=torch.long, device=device)

eos_id = tokenizer.reversed_vocab.get(b"<|endoftext|>")

out = model.generate(
    prompt,
    max_new_tokens=400,
    temperature=0.8,
    top_p=0.95,
    eos_token_id=eos_id,
)
print(tokenizer.decode(out[0].tolist()))

Once upon a time, there was a little boy named Tim. Tim was a happy dog. They both wanted to be friends. The bird said, "Hi, I am Tim. I am a bird. I am a farm. And the pig and little boy named Tom. They were playing with the same at the same to be friends. Your new friend. It was Tim and I am a wet and quiet. He was a powerful friends. Do you need to have no one was deaf to the big and laughed at the same before.
Tim and Tim and could do the fun and Tim. I am a big, but having fun and cold today. Let's turn to feel warm. Let's sleep on stage. He had a bad birds. He was a bit Tim. But the same. They played together. That means a little one, you need to be happy and I am a little too long, and a little one day. They played and didn't like playing with their favorite things that Tim and the same and I made them at noon!" Tim had never felt guilty and Tim was a baby birds. And so time. They became Tim and Tim was a ball. Tim was full of flying animals. He was near the right away. They wer

In [4]:
import torch
ckpt = torch.load("checkpoints/latest.pt", map_location="cpu")
print("iteration:", ckpt.get("iteration"))
print("loss:", ckpt.get("loss"))

sd = ckpt["model"]
print("num_layers:", max(int(k.split(".")[1]) for k in sd if k.startswith("transformerblock")) + 1)
print("d_model:", sd["embeddings.embeddings"].shape[1])
print("d_ff:", sd["transformerblock.0.ffn.w1.weight"].shape[0])
print("params:", sum(v.numel() for v in sd.values()) / 1e6, "M")

iteration: 4000
loss: None
num_layers: 4
d_model: 512
d_ff: 1344
params: 22.696448 M
